## OpenTelemetry setup

The notebook owns the tracing configuration. The shared `lib` modules only create spans,  
so callers that do not configure a provider continue to use OpenTelemetry's no-op behavior.

In [16]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()
console_span_processor = SimpleSpanProcessor(ConsoleSpanExporter())
provider.add_span_processor(console_span_processor)
trace.set_tracer_provider(provider)

Overriding of current TracerProvider is not allowed


In [17]:
from assistant import create_assistant

assitant = create_assistant()
query = "How does the agentic loop keep calling the model until it stops?"

## Q1. First trace

Run the homework query and count the span dictionaries printed by the console exporter.  
The simple RAG path creates one  parent `rag` span and one child each for `search` and `llm`.

- ❌ 1
- ✅ **3**
- ❌ 5
- ❌ 7

In [18]:
question_1_run = assitant.find_and_reply(query)

{
    "name": "search",
    "context": {
        "trace_id": "0x12f7fa0deb9ee664e9a0992dc2bc083b",
        "span_id": "0x2c279dbd3166bdab",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x0bfb35c49b44ed4a",
    "start_time": "2026-07-17T14:58:54.961850Z",
    "end_time": "2026-07-17T14:58:54.972367Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "900421e2-9328-4479-b9a5-765c74dc389c",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0x12f7fa0deb9ee664e9a0992dc2bc083b",
        "span_id": "0xbcb059ad9ebbf35e",
        "trace_state": "[]"
    },
    "kind": "SpanKind

## Q2. Capturing metrics as span attributes

Run the query again. The `llm` span includes `input_tokens`, `output_tokens`, and `cost`. Display the input-token count below and select the closest homework option.

- ✅ **700**
- ❌ 7000
- ❌ 70000
- ❌ 700000

![alt text](homework-05_question-2-3_evidence.png)

In [19]:
question_2_run = assitant.find_and_reply(query)
question_2_input_tokens = question_2_run.metrics.input_tokens
question_2_options = (700, 7000, 70000, 700000)
question_2_closest_option = min(
    question_2_options,
    key=lambda option: abs(option - question_2_input_tokens),
)
{"input_tokens": question_2_input_tokens, "closest_option": question_2_closest_option}

{
    "name": "search",
    "context": {
        "trace_id": "0x92df08becc1e784033ed0a53c76fa4b6",
        "span_id": "0x146b47ebe5d85d4d",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x76d9df93472a2edd",
    "start_time": "2026-07-17T14:58:56.507725Z",
    "end_time": "2026-07-17T14:58:56.519726Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "900421e2-9328-4479-b9a5-765c74dc389c",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0x92df08becc1e784033ed0a53c76fa4b6",
        "span_id": "0x19efaa252b3efcf0",
        "trace_state": "[]"
    },
    "kind": "SpanKind

{'input_tokens': 1557, 'closest_option': 700}

## Q3. Span timing

Each span records its start and end time. The shared RAG metrics expose the same  
LLM-call duration in seconds, making the closest range easy to identify.

> Run the LLM call two or three times and use the typical duration from the  
later calls. The first call may include cold-start overhead and may not  
represent normal subsequent-call latency.

- ❌ Under 100ms
- ❌ 100-500ms
- ✅ **500-2000ms**
- ❌ Over 2000ms

In [20]:
print(question_2_run.metrics)

AgentRunMetrics(model_call_metrics=(ModelCallMetrics(model='gpt-5.4-mini-2026-03-17', input_tokens=1557, cached_input_tokens=1280, output_tokens=8, reasoning_output_tokens=0, duration_seconds=0.6771755000081612, price=UsagePrice(input_cost_usd=0.00030375, output_cost_usd=3.6e-05), completed_at=datetime.datetime(2026, 7, 17, 14, 58, 57, 214720, tzinfo=datetime.timezone.utc)),), tool_calls_count=1, duration_seconds=0.7150664000073448)


In [21]:
question_3_llm_duration_seconds = (
    question_2_run.metrics.model_call_metrics[0].duration_seconds
)

if question_3_llm_duration_seconds < 0.1:
    question_3_closest_option = "Under 100ms"
elif question_3_llm_duration_seconds < 0.5:
    question_3_closest_option = "100-500ms"
elif question_3_llm_duration_seconds < 2.0:
    question_3_closest_option = "500-2000ms"
else:
    question_3_closest_option = "Over 2000ms"

{
    "llm_duration_seconds": question_3_llm_duration_seconds,
    "closest_option": question_3_closest_option,
}

{'llm_duration_seconds': 0.6771755000081612, 'closest_option': '500-2000ms'}

## Q4. Saving traces to SQLite

The console exporter prints completed spans but does not persist them. Create a custom `SQLiteSpanExporter` that stores each finished span in a `spans` table, then register it with the existing tracer provider. Nothing before this section can be written to SQLite because the exporter and its span processor do not exist until these cells run. The console processor remains registered, so calls from Q4 onward are visible in both the console and SQLite.

Re-run the query from Q1. Which span names appear in the `spans` table?

- ❌ Only `rag`
- ❌ `rag` and `llm`
- ✅ **`rag`, `search`, and `llm`**
- ❌ `search`, `llm`, and `judge`

In [22]:
import sqlite3
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import cast

from opentelemetry.sdk.trace import ReadableSpan
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult


class SQLiteSpanExporter(SpanExporter):
    def __init__(self, db_path: str | Path = "traces.db") -> None:
        self.connection = sqlite3.connect(db_path)
        self.connection.execute(
            """
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
            """
        )
        self.connection.commit()

    def export(self, spans: Sequence[ReadableSpan]) -> SpanExportResult:
        for span in spans:
            attributes = cast(
                Mapping[str, int | float],
                span.attributes or {},
            )

            self.connection.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attributes.get("input_tokens"),
                    attributes.get("output_tokens"),
                    attributes.get("cost"),
                ),
            )

        self.connection.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self) -> None:
        self.connection.close()

    def force_flush(self, timeout_millis: int = 30000) -> bool:
        return True

In [23]:
trace_database_path = Path("traces.db")
sqlite_exporter = SQLiteSpanExporter(trace_database_path)
sqlite_exporter.connection.execute("DELETE FROM spans")
sqlite_exporter.connection.commit()
sqlite_span_processor = SimpleSpanProcessor(sqlite_exporter)
provider.add_span_processor(sqlite_span_processor)

In [24]:
question_4_run = assitant.find_and_reply(query)

{
    "name": "search",
    "context": {
        "trace_id": "0x1995e28fc1f81f2743753c1d876d7ddb",
        "span_id": "0xd2077b0a07eb6fe6",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x3e552141a741149a",
    "start_time": "2026-07-17T14:58:57.305606Z",
    "end_time": "2026-07-17T14:58:57.314608Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "900421e2-9328-4479-b9a5-765c74dc389c",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}


{
    "name": "llm",
    "context": {
        "trace_id": "0x1995e28fc1f81f2743753c1d876d7ddb",
        "span_id": "0x437d63aa15793908",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x3e552141a741149a",
    "start_time": "2026-07-17T14:58:57.321608Z",
    "end_time": "2026-07-17T14:58:59.023221Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "input_tokens": 1557,
        "output_tokens": 65,
        "cost": 0.00059625
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "900421e2-9328-4479-b9a5-765c74dc389c",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "rag",
    "context": {
        "trace_id": "0x1995e28fc1f81f2743753c1d876d7ddb",
        "

In [25]:
with sqlite3.connect(trace_database_path) as connection:
    question_4_span_names = [
        row[0]
        for row in connection.execute(
            "SELECT DISTINCT name FROM spans ORDER BY name"
        ).fetchall()
    ]

question_4_span_names

['llm', 'rag', 'search']

## Q5. Querying trace data

The traces are now stored in SQLite. Run one more query through the traced RAG, then query the database.

The `rag` span wraps the full operation, so its duration includes both `search` and `llm`.  
Exclude `rag` and compare only the child spans to see where the time actually goes.

Using SQL or pandas, compute the total duration for each span name excluding `rag`.  

Which span type takes the most total time?
- ❌ `search`
- ✅ **`llm`**
- ❌ They're all about the same

In [26]:
question_5_run = assitant.find_and_reply(query)

{
    "name": "search",
    "context": {
        "trace_id": "0x0dd1819e0f9f671c36876e7710734bd1",
        "span_id": "0xaf91f38fa14e637d",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x49e25a0775aab3af",
    "start_time": "2026-07-17T14:58:59.068212Z",
    "end_time": "2026-07-17T14:58:59.078476Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "900421e2-9328-4479-b9a5-765c74dc389c",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}


{
    "name": "llm",
    "context": {
        "trace_id": "0x0dd1819e0f9f671c36876e7710734bd1",
        "span_id": "0xfd178cc81342b5fb",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x49e25a0775aab3af",
    "start_time": "2026-07-17T14:58:59.084476Z",
    "end_time": "2026-07-17T14:58:59.690580Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "input_tokens": 1557,
        "output_tokens": 8,
        "cost": 0.00033975
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "900421e2-9328-4479-b9a5-765c74dc389c",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "rag",
    "context": {
        "trace_id": "0x0dd1819e0f9f671c36876e7710734bd1",
        "s

In [27]:
import pandas as pd

with sqlite3.connect(trace_database_path) as connection:
    question_5_span_rows: list[tuple[str, int, int]] = connection.execute(
        """
        SELECT name, start_time, end_time
        FROM spans
        WHERE name IN ('llm', 'search')
        """
    ).fetchall()

question_5_spans_df = pd.DataFrame(
    question_5_span_rows,
    columns=["name", "start_time", "end_time"],
)

question_5_spans_df["elapsed_seconds"] = (
    question_5_spans_df["end_time"] - question_5_spans_df["start_time"]
) / 1_000_000_000
question_5_spans_df

,name,start_time,end_time,elapsed_seconds
0,search,1784300337305605700,1784300337314608500,0.009003
1,llm,1784300337321607600,1784300339023220700,1.701613
2,search,1784300339068211800,1784300339078475800,0.010264
3,llm,1784300339084475800,1784300339690579700,0.606104


In [28]:
question_5_total_elapsed_seconds_df: pd.DataFrame = question_5_spans_df.groupby(
    "name",
    as_index=False,
)[["elapsed_seconds"]].sum()
question_5_total_elapsed_seconds_df = question_5_total_elapsed_seconds_df.sort_values(
    "elapsed_seconds",
    ascending=False,
)
question_5_total_elapsed_seconds_df

,name,elapsed_seconds
0,llm,2.307717
1,search,0.019267


## Q6. Token stability across runs

A dashboard can show whether the same query produces a stable number of input tokens.  
Stable input-token counts indicate that the RAG retrieves consistent context;  
large differences may indicate unstable search results.

Run the same query from Q1 three more times, then retrieve the input-token count for  
every `llm` span stored in SQLite. The token count is already captured as a span attribute,  
so it only needs to be read from the database.

How much do the input tokens vary across the runs?

- ✅ **They're identical**
- ❌ Within 10% of each other
- ❌ Within 50% of each other
- ❌ They vary more than 50%

In [29]:
for _ in range(3):
    assitant.find_and_reply(query)

{
    "name": "search",
    "context": {
        "trace_id": "0x1465170c00c14c98463c4ef3afd77573",
        "span_id": "0x947684bdedf31877",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xa1cc063b2aea872a",
    "start_time": "2026-07-17T14:58:59.765979Z",
    "end_time": "2026-07-17T14:58:59.776972Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "900421e2-9328-4479-b9a5-765c74dc389c",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0x1465170c00c14c98463c4ef3afd77573",
        "span_id": "0xc0826a8edb76904f",
        "trace_state": "[]"
    },
    "kind": "SpanKind

In [30]:
with sqlite3.connect(trace_database_path) as connection:
    question_6_llm_rows: list[tuple[int, int]] = connection.execute(
        """
        SELECT
            ROW_NUMBER() OVER (ORDER BY start_time) AS call_number,
            input_tokens
        FROM spans
        WHERE name = 'llm'
        ORDER BY start_time
        """
    ).fetchall()

question_6_input_tokens_df = pd.DataFrame(
    question_6_llm_rows,
    columns=["call_number", "input_tokens"],
)
question_6_input_tokens_df

,call_number,input_tokens
0,1,1557
1,2,1557
2,3,1557
3,4,1557
4,5,1557
